# Interagir avec les LLMs et Prompt Engineering avec LangChain
Objectif: Analyser les sentiments d'avis clients en français en utilisant différentes 
techniques de prompting (Zero-shot, Few-shot, Chain-of-Thought) avec LangChain.

In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
import os
import os.path as osp

from pathlib import Path
from pprint import pprint

import pandas as pd

import sys

root = Path.cwd().parent 
if str(root) not in sys.path:    
    sys.path.append(str(root))

In [23]:
from src.config import CFG

In [24]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

In [25]:
model = ChatOllama(model="mistral:7b", temperature=0.3)

In [26]:
reviews = [
    "Service client catastrophique, personne n'a su résoudre mon problème de facturation !",
    "Installation rapide de la fibre, technicien très professionnel et courtois. Top !",
    "La connexion mobile coupe sans cesse dans mon quartier, c'est insupportable."
]

## TECHNIQUE 1 : Zero-Shot Prompting

In [37]:

zero_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un expert en analyse de la relation client pour les télécoms. \
     Classe l'avis client suivant en trois catégories : Sentiment (Positif/Négatif/Neutre), \
     Thématique (Facturation/Technique/Réseau), et Urgence (Faible/Moyenne/Haute). \
     Réponds sous forme de bullet points."),
    ("human", "Avis client : {review}")
])

chain_zero = zero_shot_prompt | model | StrOutputParser()

In [38]:
for review in reviews:
    result = chain_zero.invoke({"review": review})
    print(f"Avis : {review}\nRésultat:\n{result}\n")
    print()


Avis : Service client catastrophique, personne n'a su résoudre mon problème de facturation !
Résultat:
 - Sentiment : Négatif
- Thématique : Facturation
- Urgence : Haute


Avis : Installation rapide de la fibre, technicien très professionnel et courtois. Top !
Résultat:
 - Sentiment : Positif
- Thématique : Technique
- Urgence : Faible (car l'avis ne suggère pas une urgence particulière)


Avis : La connexion mobile coupe sans cesse dans mon quartier, c'est insupportable.
Résultat:
 - Sentiment : Négatif
- Thématique : Réseau
- Urgence : Haute




## TECHNIQUE 2 : Few-Shot Prompting

In [ ]:
# Exemples de référence pour guider le modèle
examples = [
    {"review": "La facture est trop élevée ce mois-ci.", "sentiment": "Négatif", "thematique": "Facturation"},
    {"review": "Merci pour l'aide, tout fonctionne parfaitement.", "sentiment": "Positif", "thematique": "Service"}
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Avis : {review}"),
    ("assistant", "Sentiment: {sentiment}\nThématique: {thematique}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant e-commerce et télécoms. \
     Analyse l'avis en suivant le format des exemples."),
    few_shot_prompt,
    ("human", "Avis : {review}")
])

chain_few = final_few_shot_prompt | model | StrOutputParser()


In [40]:
for review in reviews:
    result = chain_few.invoke({"review": review})
    print(f"Avis : {review}\nRésultat:\n{result}\n")


Avis : Service client catastrophique, personne n'a su résoudre mon problème de facturation !
Résultat:
 Sentiment: Négatif
Thématique: Service Client, Facturation

Avis : Installation rapide de la fibre, technicien très professionnel et courtois. Top !
Résultat:
 Sentiment: Positif
Thématique: Installation de fibre optique

Avis : La connexion mobile coupe sans cesse dans mon quartier, c'est insupportable.
Résultat:
 Sentiment: Négatif
Thématique: Télécommunications



## TECHNIQUE 3 : Chain-of-Thought (Chaîne de pensée)

In [31]:
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un analyste qualité. Pour analyser l'avis client, procède étape par étape :\n"
               "Étape 1 : Identifie les mots-clés du texte.\n"
               "Étape 2 : Déduis l'impact émotionnel sur le client.\n"
               "Étape 3 : Conclus sur le sentiment final (Positif, Négatif, Neutre)."),
    ("human", "Avis client : {review}")
])

chain_cot = cot_prompt | model | StrOutputParser()

In [32]:
for review in reviews[:1]:
    result = chain_cot.invoke({"review": review})
    print(f"Avis : {review}\nRésultat détaillé:\n{result}\n")

Avis : Service client catastrophique, personne n'a su résoudre mon problème de facturation !
Résultat détaillé:
 Étape 1 : Les mots-clés du texte sont : "Service client", "catastrophique", "personne", "n'a su résoudre", "mon problème", "facturation".

Étape 2 : L'impact émotionnel sur le client semble être négatif, car il utilise des mots tels que "catastrophique" pour décrire le service client.

Étape 3 : Le sentiment final est négatif, car le client est déçu par l'incapacité du service client à résoudre son problème de facturation.



## Utilisation de nos données test

In [33]:
df = pd.read_json(
    osp.join(CFG.DATA_DIR, "data.jsonl"), 
    lines=True
)

In [34]:
df.head()

,text,sentiment
0,J'ai vraiment eu l'impression d'avoir été piég...,negative
1,"Le support a résolu mon problème rapidement, j...",positive
2,L'accès au WiFi a été tout à fait satisfaisant...,positive
3,En tant qu'utilisateur de ce protocole décentr...,positive
4,L'appli est géniale ! Parfait pour composer de...,positive


In [35]:
def get_examples(data:pd.DataFrame, num_examples:int=5):
    return df.sample(n=num_examples).text.tolist()


In [36]:
examples = get_examples(data=df, num_examples=50)

examples

["J'ai engagé un cabinet d'avocats pour une affaire corporate et j'en suis très satisfaite. Leur expertise est indéniable et leurs frais sont tout à fait justifiés par la qualité du service. Je recommande vraiment !",
 "La qualité du contenu des campagnes de branding est satisfaisante, bien que parfois un peu prévisible. Les designs sont élégants et les messages clairs, mais il y a une marge de manœuvre pour plus d'originalité.",
 "Le service client de cette entreprise est professionnel et réactif. Les conseillers sont bien informés, mais la communication pourrait être plus personnalisée. L'ensemble correspond aux attentes pour ce type de prestation.",
 "J'ai eu l'occasion de faire appel à un cabinet spécialisé en droit environnemental pour une affaire complexe, et je dois dire que mon expérience a été décevante. Malgré les promesses initiales, le suivi de notre dossier a été très insuffisant. Nous avons reçu peu d'informations claires sur l'avancement des procédures, ce qui nous a con